In [ ]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from datetime import datetime
import sys


In [ ]:
# import
%run ../configs/process_logger


In [ ]:
#import
%run ../configs/catalog_config


In [ ]:
def process_brnet_DimProduct(ExecutionHeaderID, ExecutionDetailID, BusinessDate, sourcesystem, logger):
    """
    Processes BRNET source data into edw.DimProduct.
    CATALOG is used as closure variable from Cell 1.
    Only ProductId, ProductName, ProductType are populated from source;
    all other business columns are NULL. Hash = those 3 columns only.
    """

    # -------------------------------------------------------------------------
    # STEP 1 — Staging Table
    # -------------------------------------------------------------------------
    logger.log_step("Step 1: Creating mb_brnet_stg_DimProduct staging table", "EXTRACT")

    spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.edw_stg.mb_brnet_stg_DimProduct")

    spark.sql(f"""
        CREATE OR REPLACE TABLE {CATALOG}.edw_stg.mb_brnet_stg_DimProduct
        USING DELTA
        AS
        SELECT
            'BRNET'                             AS SourceSystemName,

            -- Metadata columns
            p.HKC_ETLMasterExecutionId,
            p.HKC_ETLDetailExecutionId,
            p.HKC_EDWSourceSystemID,
            current_timestamp() AS HKC_EDWLoadDate,
            current_timestamp() AS HKC_SCD2StartDate,
            current_timestamp() AS HKC_SCD2EndDate,
            1 AS HKC_SCD2RecordActiveStatus,
            p.DqStatus                          AS DQStatus,
            p.DqId                              AS DQId,

            -- ===== Only these three carry source values =====
            p.ProductID                         AS ProductId,
            p.Description                       AS ProductName,        -- CONFIRM: product name col
            p.Description                       AS ProductType,        -- CONFIRM: type col (defaulted to name)

            -- SHA1 hash — SAME 3-col list as the ACTIONFLAG comparison below
            CAST(sha1(CONCAT(
                COALESCE(CAST(p.ProductID   AS STRING), ''), '|',
                COALESCE(CAST(p.Description AS STRING), ''), '|',
                COALESCE(CAST(p.Description AS STRING), '')      -- ProductType
            )) AS BINARY)                       AS HASHBYTESSHA1

        FROM (
            SELECT * FROM (
                SELECT *,
                    ROW_NUMBER() OVER (
                        PARTITION BY ProductID
                        ORDER BY COALESCE(ModifiedOn, CreatedOn) DESC
                    ) AS rn
                FROM stg_brnet.t_product_inc_full
            ) t WHERE t.rn = 1
        ) p
    """)

    # spark.sql(f"SELECT * FROM {CATALOG}.edw_stg.mb_brnet_stg_DimProduct").display()

    # -------------------------------------------------------------------------
    # STEP 2 — Incremental Table with ACTIONFLAG
    # -------------------------------------------------------------------------
    logger.log_step("Step 2: Creating incremental table with ACTIONFLAG", "TRANSFORMATION")

    spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.edw_stg.mb_brnet_DimProduct_inc")

    spark.sql(f"""
        CREATE OR REPLACE TABLE {CATALOG}.edw_stg.mb_brnet_DimProduct_inc
        USING DELTA
        AS
        SELECT
            -- Metadata columns
            CAST(stg.HKC_ETLMasterExecutionId   AS INT)         AS HKC_ETLMasterExecutionId,
            CAST(stg.HKC_ETLDetailExecutionId   AS INT)         AS HKC_ETLDetailExecutionId,
            CAST(stg.HKC_EDWSourceSystemID      AS INT)         AS HKC_EDWSourceSystemID,
            CAST(stg.HKC_EDWLoadDate            AS TIMESTAMP)   AS HKC_EDWLoadDate,
            CAST(stg.HKC_SCD2StartDate          AS TIMESTAMP)   AS HKC_SCD2StartDate,
            CAST(stg.HKC_SCD2EndDate            AS TIMESTAMP)   AS HKC_SCD2EndDate,
            CAST(1                              AS INT)         AS HKC_SCD2RecordActiveStatus,
            CAST(stg.DQId                       AS STRING)      AS DQId,
            CAST(stg.DQStatus                   AS STRING)      AS DQStatus,

            -- Mapped business columns
            CAST(NULL                           AS BIGINT)      AS ProductKey,
            CAST(NULL                           AS BIGINT)      AS SubProductId,
            CAST(NULL                           AS STRING)      AS SubProductCode,
            CAST(NULL                           AS STRING)      AS SubProductName,
            CAST(NULL                           AS INT)         AS SubProductActiveFlag,
            CAST(stg.ProductId                  AS BIGINT)      AS ProductId,
            CAST(NULL                           AS STRING)      AS ProductCode,
            CAST(stg.ProductName                AS STRING)      AS ProductName,
            CAST(NULL                           AS INT)         AS ProductActiveFlag,
            CAST(NULL                           AS STRING)      AS LineofBusiness,
            CAST(NULL                           AS STRING)      AS StdProductName,
            CAST(NULL                           AS STRING)      AS StdSubProductName,
            CAST(NULL                           AS STRING)      AS StdProductGroup,
            CAST(NULL                           AS STRING)      AS StdProductEntity,
            CAST(NULL                           AS STRING)      AS StdProductDivision,
            CAST(NULL                           AS STRING)      AS ProductCategory,
            CAST(stg.ProductType                AS STRING)      AS ProductType,
            CAST(NULL                           AS DATE)        AS ProductStartDate,
            CAST(NULL                           AS DATE)        AS ProductEndDate,
            CAST(NULL                           AS STRING)      AS CurrencyInd,
            CAST(NULL                           AS DECIMAL(20,6)) AS MinLoan,
            CAST(NULL                           AS DECIMAL(20,6)) AS MaxLoan,
            CAST(NULL                           AS DECIMAL(20,6)) AS MinDeposit,
            CAST(NULL                           AS DECIMAL(20,6)) AS MaxDeposit,
            CAST(NULL                           AS DECIMAL(20,6)) AS MinWithdrawal,
            CAST(NULL                           AS DECIMAL(20,6)) AS MaxWithdrawal,
            CAST(NULL                           AS INT)         AS MinTerm,
            CAST(NULL                           AS INT)         AS MaxTerm,
            CAST(NULL                           AS DECIMAL(20,6)) AS FineRate,
            CAST(NULL                           AS STRING)      AS CGLCpnt1Cr,
            CAST(NULL                           AS STRING)      AS CGLCpnt2Cr,
            CAST(NULL                           AS STRING)      AS CGLCpnt1Dr,
            CAST(NULL                           AS STRING)      AS CGLCpnt2Dr,
            CAST(NULL                           AS STRING)      AS BasmBaseID,
            CAST(NULL                           AS STRING)      AS BasmRateID,
            CAST(NULL                           AS STRING)      AS RepayFrequencyID,
            CAST(NULL                           AS STRING)      AS IntRepayFrequency,
            CAST(NULL                           AS STRING)      AS LeaseLoanType,
            CAST(NULL                           AS STRING)      AS RoPrint,
            CAST(NULL                           AS STRING)      AS RoSecured,
            CAST(NULL                           AS INT)         AS CapitalizationDay,
            CAST(NULL                           AS INT)         AS IntDaysYr,
            CAST(NULL                           AS DECIMAL(20,6)) AS HighInt,
            CAST(NULL                           AS DECIMAL(20,6)) AS LowInt,
            CAST(NULL                           AS DECIMAL(20,6)) AS WriteOffMaximum,
            CAST(NULL                           AS STRING)      AS WriteOffProduct,
            CAST(NULL                           AS DECIMAL(20,6)) AS PenalIntRate1,
            CAST(NULL                           AS DECIMAL(20,6)) AS PenalIntRate2,
            CAST(NULL                           AS DECIMAL(20,6)) AS MinDiscPeriod,
            CAST(NULL                           AS STRING)      AS DiscTrmBasis,
            CAST(NULL                           AS STRING)      AS NegRateIndicatorID,
            CAST(NULL                           AS STRING)      AS AccruedMethodID,
            CAST(NULL                           AS STRING)      AS FineMethodID,
            CAST(NULL                           AS STRING)      AS EarlyClsChgApp,
            CAST(NULL                           AS STRING)      AS EarlyClosureBasis,
            CAST(NULL                           AS DECIMAL(20,6)) AS Disc1MinAmount,
            CAST(NULL                           AS DECIMAL(20,6)) AS Disc1MaxAmount,
            CAST(NULL                           AS DECIMAL(20,6)) AS Disc2MinAmount,
            CAST(NULL                           AS DECIMAL(20,6)) AS Disc2MaxAmount,
            CAST(NULL                           AS STRING)      AS PaymentMethodID,
            CAST(NULL                           AS STRING)      AS DistributionSubvention,
            CAST(NULL                           AS STRING)      AS DistributionSubventionMode,
            CAST(NULL                           AS STRING)      AS CERSAIApplicable,
            CAST(NULL                           AS DECIMAL(20,6)) AS CERSAIAmount,
            CAST(NULL                           AS STRING)      AS MOEApplicable,
            CAST(NULL                           AS DECIMAL(20,6)) AS MOEAmount,
            CAST(NULL                           AS STRING)      AS CERSAICancTermApp,
            CAST(NULL                           AS DECIMAL(20,6)) AS CERSAICancTermAmount,
            CAST(NULL                           AS STRING)      AS GraceApplicable,
            CAST(NULL                           AS DECIMAL(20,6)) AS PFApplyRate,
            CAST(NULL                           AS STRING)      AS AgeLimitCustomer,
            CAST(NULL                           AS DECIMAL(20,6)) AS RetrievalChargeAmount,
            CAST(NULL                           AS DECIMAL(20,6)) AS RetrievalChargeRate,
            CAST(NULL                           AS STRING)      AS CollateralRequired,
            CAST(NULL                           AS STRING)      AS HODisbFlag,
            CAST(NULL                           AS STRING)      AS PDDReqFlag,
            CAST(NULL                           AS STRING)      AS AmortizationMethodID,
            CAST(NULL                           AS STRING)      AS AmortizationApplicableFlg,
            CAST(NULL                           AS STRING)      AS AutoAdjustFlag,
            CAST(NULL                           AS STRING)      AS BussGroupID,
            CAST(NULL                           AS STRING)      AS CSFDSubProductSK,
            CAST(NULL                           AS STRING)      AS CSFDProductSK,
            CAST(NULL                           AS STRING)      AS PLHColumn1,
            CAST(NULL                           AS STRING)      AS PLHColumn2,
            CAST(NULL                           AS STRING)      AS PLHColumn3,
            CAST(NULL                           AS STRING)      AS PLHColumn4,
            CAST(NULL                           AS STRING)      AS PLHColumn5,
            CAST(stg.HASHBYTESSHA1              AS BINARY)      AS HASHBYTESSHA1,

            -- ACTIONFLAG — edw-side hash uses the IDENTICAL 3-col list as STG.
            -- Key is ProductId only (SubProductId is NULL for this load).
            CASE
                WHEN edw.ProductId IS NULL THEN 'I'
                WHEN edw.ProductId IS NOT NULL
                     AND stg.HASHBYTESSHA1 <> CAST(sha1(CONCAT(
                         COALESCE(CAST(edw.ProductId    AS STRING), ''), '|',
                         COALESCE(CAST(edw.ProductName  AS STRING), ''), '|',
                         COALESCE(CAST(edw.ProductType  AS STRING), '')
                     )) AS BINARY) THEN 'U'
                ELSE 'R'
            END AS ACTIONFLAG

        FROM {CATALOG}.edw_stg.mb_brnet_stg_DimProduct stg
        LEFT JOIN {CATALOG}.edw.DimProduct edw
            ON CAST(stg.ProductId AS BIGINT) = edw.ProductId
            AND edw.SubProductId IS NULL
            AND edw.HKC_EDWSourceSystemID    = stg.HKC_EDWSourceSystemID
    """)

    # spark.sql(f"SELECT * FROM {CATALOG}.edw_stg.mb_brnet_DimProduct_inc").display()

    # -------------------------------------------------------------------------
    # STEP 3 — Counts
    # -------------------------------------------------------------------------
    logger.log_step("Step 3: Counting Insert, Update and Reject records", "TRANSFORMATION")

    counts_df = spark.sql(f"""
        SELECT
            SUM(CASE WHEN ACTIONFLAG = 'I' THEN 1 ELSE 0 END) AS insert_count,
            SUM(CASE WHEN ACTIONFLAG = 'U' THEN 1 ELSE 0 END) AS update_count,
            SUM(CASE WHEN ACTIONFLAG = 'R' THEN 1 ELSE 0 END) AS reject_count
        FROM {CATALOG}.edw_stg.mb_brnet_DimProduct_inc
    """).collect()[0]

    insert_count = counts_df["insert_count"] or 0
    update_count = counts_df["update_count"] or 0
    reject_count = counts_df["reject_count"] or 0

    logger.log_step(f"Insert: {insert_count}, Update: {update_count}, Reject (No Change): {reject_count}", "TRANSFORMATION")

    # -------------------------------------------------------------------------
    # STEP 4 — DELETE + INSERT
    # -------------------------------------------------------------------------
    logger.log_step("Step 4: DELETE existing + INSERT into DimProduct", "LOAD")

    delete_count = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM {CATALOG}.edw.DimProduct edw
        WHERE EXISTS (
            SELECT 1
            FROM {CATALOG}.edw_stg.mb_brnet_DimProduct_inc inc
            WHERE inc.ACTIONFLAG = 'U'
              AND inc.ProductId  = edw.ProductId
              AND edw.SubProductId IS NULL
              AND edw.HKC_EDWSourceSystemID = inc.HKC_EDWSourceSystemID
        )
    """).collect()[0]["cnt"]

    logger.log_step(f"Deleted {delete_count} records for update", "LOAD")

    spark.sql(f"""
        DELETE FROM {CATALOG}.edw.DimProduct
        WHERE EXISTS (
            SELECT 1
            FROM {CATALOG}.edw_stg.mb_brnet_DimProduct_inc inc
            WHERE inc.ACTIONFLAG = 'U'
              AND inc.ProductId  = {CATALOG}.edw.DimProduct.ProductId
              AND {CATALOG}.edw.DimProduct.SubProductId IS NULL
              AND {CATALOG}.edw.DimProduct.HKC_EDWSourceSystemID = inc.HKC_EDWSourceSystemID
        )
    """)

    spark.sql(f"""
        INSERT INTO {CATALOG}.edw.DimProduct (
            HKC_ETLMasterExecutionId, HKC_ETLDetailExecutionId, HKC_EDWSourceSystemID,
            HKC_EDWLoadDate, HKC_SCD2StartDate, HKC_SCD2EndDate, HKC_SCD2RecordActiveStatus,
            DQId, DQStatus,
            SubProductId, SubProductCode, SubProductName, SubProductActiveFlag,
            ProductId, ProductCode, ProductName, ProductActiveFlag, LineofBusiness,
            StdProductName, StdSubProductName, StdProductGroup, StdProductEntity, StdProductDivision,
            ProductCategory, ProductType, ProductStartDate, ProductEndDate, CurrencyInd,
            MinLoan, MaxLoan, MinDeposit, MaxDeposit, MinWithdrawal, MaxWithdrawal,
            MinTerm, MaxTerm, FineRate, CGLCpnt1Cr, CGLCpnt2Cr, CGLCpnt1Dr, CGLCpnt2Dr,
            BasmBaseID, BasmRateID, RepayFrequencyID, IntRepayFrequency, LeaseLoanType,
            RoPrint, RoSecured, CapitalizationDay, IntDaysYr, HighInt, LowInt,
            WriteOffMaximum, WriteOffProduct, PenalIntRate1, PenalIntRate2, MinDiscPeriod,
            DiscTrmBasis, NegRateIndicatorID, AccruedMethodID, FineMethodID, EarlyClsChgApp,
            EarlyClosureBasis, Disc1MinAmount, Disc1MaxAmount, Disc2MinAmount, Disc2MaxAmount,
            PaymentMethodID, DistributionSubvention, DistributionSubventionMode,
            CERSAIApplicable, CERSAIAmount, MOEApplicable, MOEAmount, CERSAICancTermApp,
            CERSAICancTermAmount, GraceApplicable, PFApplyRate, AgeLimitCustomer,
            RetrievalChargeAmount, RetrievalChargeRate, CollateralRequired, HODisbFlag,
            PDDReqFlag, AmortizationMethodID, AmortizationApplicableFlg, AutoAdjustFlag,
            BussGroupID, CSFDSubProductSK, CSFDProductSK,
            PLHColumn1, PLHColumn2, PLHColumn3, PLHColumn4, PLHColumn5, HASHBYTESSHA1
        )
        SELECT
            HKC_ETLMasterExecutionId, HKC_ETLDetailExecutionId, HKC_EDWSourceSystemID,
            HKC_EDWLoadDate, HKC_SCD2StartDate, HKC_SCD2EndDate, HKC_SCD2RecordActiveStatus,
            DQId, DQStatus,
            SubProductId, SubProductCode, SubProductName, SubProductActiveFlag,
            ProductId, ProductCode, ProductName, ProductActiveFlag, LineofBusiness,
            StdProductName, StdSubProductName, StdProductGroup, StdProductEntity, StdProductDivision,
            ProductCategory, ProductType, ProductStartDate, ProductEndDate, CurrencyInd,
            MinLoan, MaxLoan, MinDeposit, MaxDeposit, MinWithdrawal, MaxWithdrawal,
            MinTerm, MaxTerm, FineRate, CGLCpnt1Cr, CGLCpnt2Cr, CGLCpnt1Dr, CGLCpnt2Dr,
            BasmBaseID, BasmRateID, RepayFrequencyID, IntRepayFrequency, LeaseLoanType,
            RoPrint, RoSecured, CapitalizationDay, IntDaysYr, HighInt, LowInt,
            WriteOffMaximum, WriteOffProduct, PenalIntRate1, PenalIntRate2, MinDiscPeriod,
            DiscTrmBasis, NegRateIndicatorID, AccruedMethodID, FineMethodID, EarlyClsChgApp,
            EarlyClosureBasis, Disc1MinAmount, Disc1MaxAmount, Disc2MinAmount, Disc2MaxAmount,
            PaymentMethodID, DistributionSubvention, DistributionSubventionMode,
            CERSAIApplicable, CERSAIAmount, MOEApplicable, MOEAmount, CERSAICancTermApp,
            CERSAICancTermAmount, GraceApplicable, PFApplyRate, AgeLimitCustomer,
            RetrievalChargeAmount, RetrievalChargeRate, CollateralRequired, HODisbFlag,
            PDDReqFlag, AmortizationMethodID, AmortizationApplicableFlg, AutoAdjustFlag,
            BussGroupID, CSFDSubProductSK, CSFDProductSK,
            PLHColumn1, PLHColumn2, PLHColumn3, PLHColumn4, PLHColumn5, HASHBYTESSHA1
        FROM {CATALOG}.edw_stg.mb_brnet_DimProduct_inc
        WHERE ACTIONFLAG IN ('I', 'U')
    """)

    inserted_total = insert_count + update_count
    logger.log_step(f"Inserted {inserted_total} records (New: {insert_count}, Updated: {update_count})", "LOAD")

    # -------------------------------------------------------------------------
    # STEP 5 — Validation + Final Count
    # -------------------------------------------------------------------------
    logger.log_step("Step 5: Validation - Check for duplicate ProductId", "VALIDATION")

    src_sys_id = spark.sql(f"""
        SELECT MAX(HKC_EDWSourceSystemID) AS id
        FROM {CATALOG}.edw_stg.mb_brnet_stg_DimProduct
    """).collect()[0]["id"]

    dup_df = spark.sql(f"""
        SELECT ProductId, COUNT(*) AS cnt
        FROM {CATALOG}.edw.DimProduct
        WHERE HKC_EDWSourceSystemID = {src_sys_id}
          AND SubProductId IS NULL
        GROUP BY ProductId
        HAVING COUNT(*) > 1
    """)

    if dup_df.count() > 0:
        dup_df.display()
        raise Exception(f"Duplicate ProductId found in edw.DimProduct for HKC_EDWSourceSystemID = {src_sys_id}")

    final_count = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM {CATALOG}.edw.DimProduct
        WHERE HKC_EDWSourceSystemID = {src_sys_id}
          AND SubProductId IS NULL
    """).collect()[0]["cnt"]

    logger.log_step(
        f"BRNET completed. Final: {final_count} | Insert: {insert_count} | Update: {update_count} | Reject: {reject_count} | Deleted: {delete_count}",
        "COMPLETED"
    )

    return final_count


In [ ]:
def usp_mb_Load_DimProduct(ExecutionHeaderID, ExecutionDetailID, DestTableName, BusinessDate, sourcesystem):
    """
    Orchestrator for edw.DimProduct incremental load.

    Supported source systems: BRNET
    Sources: stg_brnet.t_grouploanscheme (gls) LEFT JOIN stg_brnet.t_product (p)
             ON gls.LoanProductID = p.ProductID
    Business Key: LoanSchemeID (SubProductId) + ProductID (ProductId)

    Parameters:
        ExecutionHeaderID : str  — ADF pipeline execution header ID
        ExecutionDetailID : str  — ADF pipeline execution detail ID
        DestTableName     : str  — Target table name
        BusinessDate      : str  — Business date (YYYY-MM-DD)
        sourcesystem      : str  — Source system name

    Returns:
        1 on success, raises exception on failure.
    """
    logger = ProcessLogger('edw_usp_mb_Load_DimProduct', 'ETL', spark)

    try:
        logger.log_start(
            ExecutionHeaderID=ExecutionHeaderID,
            ExecutionDetailID=ExecutionDetailID,
            DestTableName=DestTableName,
            BusinessDate=BusinessDate,
            sourcesystem=sourcesystem
        )

        logger.log_step(f"Starting DimProduct load for source system: {sourcesystem}", "EXTRACT")

        records_processed = 0

        if sourcesystem in ('BRNET', 'Brnet'):
            logger.log_step(f"Processing {sourcesystem} source system", "TRANSFORMATION")
            records_processed = process_brnet_DimProduct(
                ExecutionHeaderID=ExecutionHeaderID,
                ExecutionDetailID=ExecutionDetailID,
                BusinessDate=BusinessDate,
                sourcesystem=sourcesystem,
                logger=logger
            )

        else:
            error_msg = f"Unknown source system: {sourcesystem}. Valid values are: BRNET"
            logger.log_error(error_msg)
            raise ValueError(error_msg)

        logger.log_success(
            total_records_processed=records_processed,
            DestTableName=DestTableName,
            BusinessDate=BusinessDate,
            sourcesystem=sourcesystem
        )

        print("SUCCESS: 1")
        return 1

    except Exception as e:
        logger.log_error(f"Error processing source system {sourcesystem}: {str(e)}")
        print("FAILURE: 0")
        raise e


In [ ]:
CATALOG = "lakehouse_uat"

dbutils.widgets.text("ExecutionHeaderID", "1", "Execution Header ID")
dbutils.widgets.text("ExecutionDetailID", "1", "Execution Detail ID")
dbutils.widgets.text("DestTableName", "edw.DimProduct", "Destination Table Name")
dbutils.widgets.text("BusinessDate", "YYYY-MM-DD", "Business Date (YYYY-MM-DD)")
dbutils.widgets.dropdown("sourcesystem", "BRNET", ["BRNET"], "Source System")

ExecutionHeaderID = dbutils.widgets.get("ExecutionHeaderID")
ExecutionDetailID = dbutils.widgets.get("ExecutionDetailID")
DestTableName     = dbutils.widgets.get("DestTableName")
BusinessDate      = dbutils.widgets.get("BusinessDate")
sourcesystem      = dbutils.widgets.get("sourcesystem")

result = usp_mb_Load_DimProduct(
    ExecutionHeaderID=ExecutionHeaderID,
    ExecutionDetailID=ExecutionDetailID,
    DestTableName=DestTableName,
    BusinessDate=BusinessDate,
    sourcesystem=sourcesystem
)
print(result)
